# PoliMillionaire — NLP Chatbot Assignment

A chatbot that plays the *Who wants to be a PoliMillionaire?* quiz via a REST API client.

**Competitions:**
| ID | Name | Questions |
|----|------|-----------|
| 0  | Entertainment | 15 |
| 1  | Ancient History & Politics | 15 |
| 2  | Science & Nature | 15 |
| 3  | Maths | 15 |

**Architecture:**
- **LLM:** Qwen2.5-7B-Instruct (float16, greedy decoding for speed)
- **RAG:** DuckDuckGo (Entertainment) · Wikipedia (History, Science) · Wikidata SPARQL fallback (Science) · Calculator (Maths)
- **Answer extraction:** regex-based, robust fallback chain

## 0 · Setup — Google Drive mount

Mount Drive so the `millionaire_client` package and this notebook share the same directory.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
import sys

# The path where both PoliMillionaire.ipynb and millionaire_client/ live
PACKAGE_DIR = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment'

if PACKAGE_DIR not in sys.path:
    sys.path.append(PACKAGE_DIR)

print('Path configured:', PACKAGE_DIR)

## 1 · Install dependencies

Required packages beyond the standard Colab environment:

In [ ]:
%pip install -q duckduckgo-search wikipedia transformers accelerate

## 2 · Import the bot module

All game logic lives in `millionaire_bot.py`. Import it, we do.

In [ ]:
import millionaire_bot as bot

print('Imported successfully, the bot module has.')

## 3 · Load the LLM

Qwen2.5-7B-Instruct is loaded with `device_map="auto"` and `torch_dtype=float16`.
This downloads ~14 GB on first run — patience, the Force requires.

In [ ]:
bot.load_model('Qwen/Qwen2.5-7B-Instruct')

## 4 · Connect to the API

Authenticate with the PoliMillionaire server. Store your password in a Colab secret named `poli-millionaire`.

In [ ]:
from google.colab import userdata
from millionaire_client import MillionaireClient, AuthenticationError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'riccardo'           # Change to your username
PASSWORD = userdata.get('poli-millionaire')

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Welcome, {user.username}! Role: {user.role}')
except AuthenticationError as e:
    print(f'Login failed: {e}')

## 5 · Browse available competitions

In [ ]:
competitions = client.competitions.list_all()
print('=== Available Competitions ===')
for c in competitions:
    print(f'  [{c.id}] {c.name} — {c.max_levels} questions')

## 6 · Play a single competition

Choose a `COMP_ID` (0–3) and let the bot play one full game.

In [ ]:
COMP_ID = bot.COMP_HISTORY_POLITICS   # Change this to try other competitions

game = client.game.start(competition_id=COMP_ID)
print(f'Session ID: {game.session_id} | Questions: {game.state.competition.max_levels}')

game_log = bot.play_game(game, COMP_ID)
bot.print_evaluation(game_log)

## 7 · Play all four competitions

Run all competitions sequentially and aggregate results.
A complete evaluation, this produces.

In [ ]:
all_logs = []

for comp_id in [bot.COMP_ENTERTAINMENT,
                bot.COMP_HISTORY_POLITICS,
                bot.COMP_SCIENCE_NATURE,
                bot.COMP_MATHS]:
    game = client.game.start(competition_id=comp_id)
    print(f'Session {game.session_id} started for competition {comp_id}.')
    log = bot.play_game(game, comp_id)
    all_logs.append(log)

bot.print_all_evaluations(all_logs)

## 8 · Leaderboard

See where the bot ranks after playing.

In [ ]:
for comp_id in range(4):
    lb = client.leaderboard.get(competition_id=comp_id, limit=10)
    print(f'\n=== Leaderboard: {lb.competition.name} ===')
    for i, entry in enumerate(lb.entries[:10], 1):
        marker = ' <-- YOU' if entry.username == USERNAME else ''
        print(f'  {i:>2}. {entry.username:<20} ${entry.score:>10,.2f}  (Level {entry.reached_level}){marker}')

## 9 · Quick unit tests for key functions

Verify the helper functions work correctly before running a live game.

In [ ]:
# --- Test extract_answer_id ---
cases = [
    ('The answer is 2, definitely.',          0, 4, 2),
    ('I think option B is correct.',          0, 4, 1),
    ('No clue at all.',                       0, 4, 0),
    ('3_Full chain-of-thought reasoning...',  0, 4, 3),
    ('Option A seems right.',                 0, 4, 0),
]
print('=== extract_answer_id tests ===')
all_passed = True
for text, default, n_opts, expected in cases:
    got = bot.extract_answer_id(text, n_opts)
    status = '✓' if got == expected else '✗'
    if got != expected:
        all_passed = False
    print(f'  {status}  "{text[:40]}" → {got} (expected {expected})')
print('All passed!' if all_passed else 'Some failed!')

# --- Test rag_maths ---
print('\n=== rag_maths tests ===')
maths_cases = [
    'What is 15% of 200?',
    'Calculate the square root of 144.',
    'What is 2^10?',
    'What is 3 + 4 * 2?',
]
for q in maths_cases:
    result = bot.rag_maths(q)
    print(f'  Q: {q}')
    print(f'  → {result}\n')

## 10 · Save logs to JSON

Persist all game logs for offline analysis.

In [ ]:
import json

log_path = f'{PACKAGE_DIR}/game_logs.json'
with open(log_path, 'w') as f:
    json.dump(all_logs, f, indent=2)

print(f'Saved {len(all_logs)} game log(s) to {log_path}')